# 🎨 Modul 05 Hands-on: Image Segmentation & The U-Net Architecture

Pada hands-on ini, Anda akan mempraktikkan segmentasi citra tingkat piksel:
1. **The Working Blackbox**: Membangun U-Net sederhana untuk segmentasi objek geometris.
2. **Anatomical Dissection**: Encoder $\rightarrow$ Bottleneck $\rightarrow$ Decoder dengan Skip Connection antar level resolusi.
3. **The Sabotage Lab: Class Imbalance in Segmentation!** Cross-Entropy gagal pada objek tipis/kecil, sementara **Dice Loss** berhasil mempertahankan batas tepi objek secara presisi.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device aktif: {device}")

## 1. Menyiapkan Dataset Segmentasi Sintetis (Lingkaran Kecil di Background Luas)

In [ ]:
# 100 gambar biner 1x64x64 dengan lingkaran kecil (hanya ~5% piksel foreground)
N = 100
imgs = torch.zeros(N, 1, 64, 64)
masks = torch.zeros(N, 1, 64, 64)

for i in range(N):
    cx, cy = np.random.randint(20, 44, 2)
    r = np.random.randint(6, 12)
    y, x = np.ogrid[:64, :64]
    mask = ((x - cx)**2 + (y - cy)**2) <= r**2
    masks[i, 0] = torch.tensor(mask, dtype=torch.float32)
    imgs[i, 0] = masks[i, 0] + torch.randn(64, 64)*0.2

imgs, masks = imgs.to(device), masks.to(device)
print(f"Rasio piksel foreground: {(masks.sum() / masks.numel()) * 100:.2f}% (Sangat Imbalanced!)")

## 2. Arsitektur Mini U-Net dengan Skip Connections

In [ ]:
class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder (Down)
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2)  # 64 -> 32
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2)  # 32 -> 16
        
        # Bottleneck
        self.bottleneck = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU())
        
        # Decoder (Up)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)  # 16 -> 32
        self.dec2 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU())  # 32 up + 32 skip = 64
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)  # 32 -> 64
        self.dec1 = nn.Sequential(nn.Conv2d(32, 16, 3, padding=1), nn.ReLU())  # 16 up + 16 skip = 32
        self.final = nn.Conv2d(16, 1, 1)
        
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))  # Skip connection e2!
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1)) # Skip connection e1!
        return self.final(d1)

model = MiniUNet().to(device)
print("Mini U-Net berhasil diinstansiasi!")

## 3. The Sabotage Lab: BCE vs Dice Loss
BCE Loss menghitung eror per piksel. Jika 95% piksel adalah background hitam, model bisa mencapai akurasi 95% hanya dengan memprediksi background hitam untuk semua piksel!
**Dice Loss** mengukur overlap area setara F1-Score tingkat piksel.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

dice_crit = DiceLoss()
opt = optim.Adam(model.parameters(), lr=0.01)

print("Training U-Net dengan Dice Loss (30 Epochs)...")
for ep in range(1, 31):
    model.train()
    opt.zero_grad()
    pred = model(imgs)
    loss = dice_crit(pred, masks)
    loss.backward()
    opt.step()
    if ep % 10 == 0:
        print(f"Epoch {ep}/30 | Dice Loss: {loss.item():.4f} (Dice Score: {1-loss.item():.2%})")

# Visualisasi Prediksi Piksel
model.eval()
with torch.no_grad():
    test_pred = (torch.sigmoid(model(imgs[:3])) > 0.5).float()

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i in range(3):
    axes[i, 0].imshow(imgs[i, 0].cpu().numpy(), cmap="gray"); axes[i, 0].set_title("Input Image")
    axes[i, 1].imshow(masks[i, 0].cpu().numpy(), cmap="gray"); axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 2].imshow(test_pred[i, 0].cpu().numpy(), cmap="gray"); axes[i, 2].set_title("Prediksi U-Net (Dice)")
    for j in range(3): axes[i, j].axis("off")
plt.tight_layout(); plt.show()